In [1]:
# IMPORTS
from egomimic.rldb.utils import *
import torch
import numpy as np
from egomimic.utils.egomimicUtils import CameraTransforms, draw_actions
import torchvision.io as io
import os

/nethome/paphiwetsa3/flash/projects/EgoVerse/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Load dataset
root = "/nethome/paphiwetsa3/flash/projects/EgoVerse/datasets"
repo_id = "rpuns/aria_laundry_rl2"

episodes = [0, 1]
dataset = RLDBDataset(
    repo_id=repo_id, root=root, local_files_only=True, episodes=episodes, mode="sample"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
# Get metadata
print(dataset.meta.info["features"])

image_key = "observations.images.front_img_1"
actions_key = "actions_cartesian"

{'observations.state.ee_pose': {'dtype': 'float64', 'shape': (12,), 'names': ['dim_0']}, 'observations.images.front_img_1': {'dtype': 'image', 'shape': (480, 640, 3), 'names': ['channel', 'height', 'width']}, 'actions_cartesian': {'dtype': 'prestacked_float64', 'shape': (100, 12), 'names': ['chunk_length', 'action_dim']}, 'metadata.embodiment': {'dtype': 'int32', 'shape': (1,), 'names': ['dim_0']}, 'timestamp': {'dtype': 'float32', 'shape': (1,), 'names': None}, 'frame_index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'episode_index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'task_index': {'dtype': 'int64', 'shape': (1,), 'names': None}}


In [4]:
print(dataset.embodiment)

5


In [11]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)

In [12]:
camera_transforms = CameraTransforms(intrinsics_key="base", extrinsics_key="x5Nov18_3")

In [19]:
def visualize_actions(ims, actions, extrinsics, intrinsics, arm="both"):
    for b in range(actions.shape[0]):
        if actions.shape[-1] == 7 or actions.shape[-1] == 14:
            ac_type = "joints"
        elif actions.shape[-1] == 3 or actions.shape[-1] == 6:
            ac_type = "xyz"
        else:
            raise ValueError(f"Unknown action type with shape {actions.shape}")

        ims[b] = draw_actions(
            ims[b], ac_type, "Purples", actions[b], extrinsics, intrinsics, arm=arm
        )

    return ims

In [20]:
save_dir = "./visualization/"
os.makedirs(save_dir, exist_ok=True)

num_batches = 6

for i, data in enumerate(data_loader):
    if i > num_batches:
        break
    ims = (data[image_key].permute(0, 2, 3, 1).cpu().numpy() * 255.0).astype(np.uint8)
    actions = data[actions_key].cpu().numpy()
    actions = actions[:1, ...]
    ims = ims[:1, ...]
    left_actions = actions[..., :3]
    right_actions = actions[..., 7:10]
    both_actions = np.concatenate([left_actions, right_actions], axis=-1)
    print(both_actions.shape)
    ims_viz = visualize_actions(ims, both_actions, camera_transforms.extrinsics, camera_transforms.intrinsics)

    for j, im in enumerate(ims_viz):
        img_tensor = torch.from_numpy(im).permute(2, 0, 1)
        save_path = os.path.join(save_dir, f"image_{i}_{j}.png")
        io.write_png(img_tensor, save_path)

    print(f"Saved batch {i} images to {save_dir}")

(1, 100, 6)
Saved batch 0 images to ./visualization/
(1, 100, 6)
Saved batch 1 images to ./visualization/
(1, 100, 6)
Saved batch 2 images to ./visualization/
(1, 100, 6)
Saved batch 3 images to ./visualization/
(1, 100, 6)
Saved batch 4 images to ./visualization/
(1, 100, 6)
Saved batch 5 images to ./visualization/
(1, 100, 6)
Saved batch 6 images to ./visualization/
